# Training Baseline

Notebook base para entrenar modelos individuales reutilizando utilidades en `src/`.

Secciones: configuración, carga de datos, datasets/dataloaders, modelo, pérdida/optimizador, entrenamiento, evaluación y Grad-CAM.

## 1) Imports y configuración básica

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from datetime import datetime
import json

# Asegurar que la raíz del repo está en sys.path (notebook ubicado en `baseline/`)
sys.path.append(str(Path('..').resolve()))

# Importar utilidades desde src
from src import (
    set_global_seed,
    get_device,
    CONFIG,
    CATALOG_PATH,
    FITS_DIR,
    GalaxyDataset,
    build_resnet18,
    compute_classification_metrics,
    train_loop,
    generate_gradcam_visualization,
)

print('Imports OK')

## 2) Semilla y dispositivo

In [ ]:
# Config reproducibilidad y dispositivo
set_global_seed(CONFIG['seed'])
device = get_device()
print(f'Using device: {device}')

## 3) Carga del catálogo y división stratificada

- El `CONFIG` centraliza los hiperparámetros y también la semilla aleatoria usada en el pipeline.

- En esta celda usamos `CONFIG['seed']` en `train_test_split`, lo que garantiza que la división stratificada 70/15/15 sea reproducible.

- Además, en la celda anterior `set_global_seed(CONFIG['seed'])` fija los generadores de números aleatorios para PyTorch, NumPy y otras librerías compatibles. Esta es una forma de asegurar la reproducibilidad en todos los componentes que usan aleatoriedad.

- De esta forma, los datos, el orden aleatorio de los batchs y las transformaciones con componente estocástico se mantienen consistentes entre ejecuciones.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

catalog_path = Path(CATALOG_PATH)
assert catalog_path.exists(), f'Catalog not found: {catalog_path}'
df = pd.read_csv(catalog_path)
df.drop(columns=['source'], errors='ignore', inplace=True)

# Excluir registros corruptos si existe reporte procesado
report_path = Path('..') / 'data' / 'processed' / 'dataset_validation_report.csv'
if report_path.exists():
    report_df = pd.read_csv(report_path)
    invalid_ids = report_df[report_df['complete'] == False]['name'].tolist()
    df = df[~df['name'].isin(invalid_ids)].reset_index(drop=True)

# Stratified split: 70/15/15
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=CONFIG['seed'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=CONFIG['seed'])

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

## 4) Datasets y DataLoaders

- `FITS_DIR` apunta al directorio donde están almacenados los archivos FITS usados por `GalaxyDataset`. Si esos archivos no están disponibles en esa ruta, la construcción del dataset falla y no se podrá cargar ninguna imagen para entrenamiento, validación o test.

- En esta sección definimos los datasets y los `DataLoader`, pero la información de cómo transformar la imagen (`target_shape`), el tamaño de batch, el número de workers y si se aplican aumentos queda centralizada en `CONFIG`. Esto facilita:
    - cambiar la configuración desde un solo lugar,
    - reproducir experimentos con los mismos parámetros,
    - y adaptar el pipeline a otros modelos o conjuntos de datos sin modificar múltiples celdas.

In [ ]:
from torch.utils.data import DataLoader

train_dataset = GalaxyDataset(train_df, FITS_DIR, target_shape=CONFIG['image_size'], augment=True)
val_dataset = GalaxyDataset(val_df, FITS_DIR, target_shape=CONFIG['image_size'], augment=False)
test_dataset = GalaxyDataset(test_df, FITS_DIR, target_shape=CONFIG['image_size'], augment=False)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])

print('Datasets and DataLoaders ready')

## 5) Construcción del modelo (ResNet18 baseline)

- En esta sección construimos el modelo usando la función `build_resnet18` de `src.models`, que actúa como el punto de homologación del pipeline.
- El objetivo es que la definición de la arquitectura quede encapsulada en esa función y que el resto del notebook no dependa de detalles específicos de ResNet18.
- Por eso `build_resnet18` recibe parámetros genéricos como `num_classes`, `pretrained`, `freeze_backbone` y `device`, y devuelve tanto el modelo como información sobre parámetros entrenables y congelados.
- Con este enfoque, cambiar a otra arquitectura debe ser un cambio mínimo: basta con reemplazar la llamada a `build_resnet18` por otra función de construcción (`build_efficientnet`, `build_densenet`, etc.) que respete la misma interfaz.
- De esta forma se mantiene la reproducibilidad y la reutilización del notebook, porque el entrenamiento, la evaluación y la generación de Grad-CAM pueden seguir usando el mismo flujo sin modificar más celdas.

In [ ]:
# Construir ResNet18 con las utilidades de src.models
model, trainable_params, frozen_params = build_resnet18(num_classes=CONFIG['num_classes'], pretrained=True, freeze_backbone=CONFIG.get('freeze_backbone', True), device=device)
print(f'Trainable params: {trainable_params:,} | Frozen params: {frozen_params:,}')

# Verificación rápida de forward con un batch (si hay datos suficientes)
try:
    sample_images, _ = next(iter(train_loader))
    sample_images = sample_images.to(device)
    with torch.no_grad():
        outputs = model(sample_images[:2])
    print('Forward OK, outputs shape:', outputs.shape)
except Exception as e:
    print('Forward test skipped:', e)

## 6) Pérdida, pesos de clase y optimizador

- En esta sección se documenta la definición de la función de pérdida y el optimizador como componentes genéricos del pipeline.

- Los pesos de clase se calculan a partir de `train_df`, lo que permite compensar el desequilibrio de clases sin depender de la arquitectura del modelo.

- El criterio usa `nn.CrossEntropyLoss(weight=class_weights)`, una opción estándar para clasificación multiclase/binaria que se puede reutilizar con cualquier red que produzca logits.

- El optimizador se construye a partir de `filter(lambda p: p.requires_grad, model.parameters())`, por lo que solo afecta a los parámetros entrenables del modelo actual y es compatible con otros modelos que reemplacen a `ResNet18`.

- Los hiperparámetros (`learning_rate`, `weight_decay`, `num_classes`) se toman desde `CONFIG`, lo que centraliza la configuración y facilita experimentar con distintas arquitecturas manteniendo la misma lógica de entrenamiento.

- La sección actúa como una capa de homologación del notebook: separa la lógica de optimización y pérdida de los detalles del modelo, permitiendo reutilizar el mismo flujo con otras arquitecturas de redes neuronales simplemente cambiando la construcción del modelo en la sección anterior.

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Calcular pesos de clase a partir de train_df
from collections import Counter
counts = Counter(train_df['label'].tolist())
num_samples = len(train_df)
num_classes = CONFIG['num_classes']
class_weights = [num_samples / (num_classes * counts.get(i, 1)) for i in range(num_classes)]
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])

print('Criterion and optimizer ready')

## 7) Entrenamiento

- Esta sección ejecuta el entrenamiento a través de `train_loop` usando los datasets, el optimizador y la pérdida definidos anteriormente, sin acoplarse a una arquitectura concreta.

- Al tomar `epochs` y otros hiperparámetros desde `CONFIG`, el flujo de entrenamiento se mantiene genérico y fácil de reutilizar con otros modelos.

- El guardado de checkpoint y metadatos (`model_state_dict`, `optimizer_state_dict`, `history`, `training_params.json`, `history.json`) se hace de forma independiente del modelo, lo que facilita la comparación y recuperación de experimentos.

- La extracción de información como `trainable_params`, `frozen_params`, `class_weights` y rutas de datos en un JSON contribuye a que el notebook sea un pipeline reproducible y portable.

- En conjunto, esta sección funciona como una capa de homologación: permite cambiar la arquitectura (por ejemplo, `build_efficientnet`, `build_densenet`, etc.) sin necesidad de modificar la lógica de entrenamiento y guardado de resultados.

In [ ]:
model_name = 'resnet18_baseline'
model_dir = Path('data') / 'models' / model_name
model_dir.mkdir(parents=True, exist_ok=True)

epochs = CONFIG['epochs']
model, history = train_loop(model, train_loader, val_loader, criterion, optimizer, device, epochs=epochs)

checkpoint_path = model_dir / 'checkpoint.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history,
    'config': CONFIG,
    'class_weights': class_weights.cpu().tolist(),
    'trainable_params': trainable_params,
    'frozen_params': frozen_params,
    'created_at': datetime.now(tz="America Mexico City").isoformat(),
}, checkpoint_path)

def make_json_serializable(value):
    if isinstance(value, torch.Tensor):
        return value.cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {k: make_json_serializable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_serializable(v) for v in value]
    return value

training_meta = {
    'model_name': model_name,
    'created_at': datetime.now(tz="America Mexico City").isoformat(),
    'epochs': epochs,
    'config': CONFIG,
    'trainable_params': trainable_params,
    'frozen_params': frozen_params,
    'dataset_split': {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)},
    'catalog_path': str(catalog_path),
    'fits_dir': str(FITS_DIR),
    'class_weights': class_weights.cpu().tolist(),
    'checkpoint_file': checkpoint_path.name,
    'history_file': 'history.json',
}

with open(model_dir / 'training_params.json', 'w') as f:
    json.dump(make_json_serializable(training_meta), f, indent=2)

with open(model_dir / 'history.json', 'w') as f:
    json.dump(make_json_serializable(history), f, indent=2)

print(f'Checkpoint saved: {checkpoint_path}')
print(f'Model artifacts saved under: {model_dir}')

## 8) Evaluación final en Test Set

In [ ]:
# Inferencia sobre test set
model.eval()
all_labels = []
all_preds = []
all_probs = []

# Preparar carpeta de evaluación y envolver la función de métricas para que guarde resultados al ser llamada
eval_dir = model_dir / 'evaluation'
eval_dir.mkdir(parents=True, exist_ok=True)

# Intentar obtener nombres de muestra en el mismo orden que test_loader (shuffle=False)
sample_names = None
if hasattr(test_dataset, 'df'):
    sample_names = test_dataset.df['name'].tolist()
elif hasattr(test_dataset, 'dataframe'):
    sample_names = test_dataset.dataframe['name'].tolist()
elif 'name' in test_df.columns:
    sample_names = test_df['name'].tolist()

_orig_compute_metrics = compute_classification_metrics

def _compute_and_save(y_true_, y_pred_, y_probs_):
    metrics = _orig_compute_metrics(y_true_, y_pred_, y_probs_)

    # Guardar arrays
    np.save(eval_dir / 'y_true.npy', y_true_)
    np.save(eval_dir / 'y_pred.npy', y_pred_)
    np.save(eval_dir / 'y_probs.npy', y_probs_)

    # Guardar CSV con predicciones si tenemos los nombres y coinciden en longitud
    try:
        if sample_names is not None and len(sample_names) == len(y_true_):
            preds_df = pd.DataFrame({
                'name': sample_names,
                'label': y_true_,
                'pred': y_pred_,
                'prob': y_probs_,
            })
            preds_df.to_csv(eval_dir / 'predictions.csv', index=False)
    except Exception:
        pass

    # Guardar métricas como JSON
    with open(eval_dir / 'metrics.json', 'w') as f:
        json.dump(make_json_serializable(metrics), f, indent=2)

    return metrics

# Reemplazar la función por la versión que además guarda los artefactos
compute_classification_metrics = _compute_and_save
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

y_true = np.array(all_labels)
y_pred = np.array(all_preds)
y_probs = np.array(all_probs)

metrics = compute_classification_metrics(y_true, y_pred, y_probs)
print('Test metrics:')
for k, v in metrics.items():
    print(f'{k}: {v}')

In [ ]:
learning_curves = {
    'epochs': list(range(1, len(history['train_loss']) + 1)),
    'train_loss': history['train_loss'],
    'val_loss': history['val_loss'],
    'train_f1': history['train_f1'],
    'val_f1': history['val_f1'],
}

with open(model_dir / 'learning_curves.json', 'w') as f:
    json.dump(make_json_serializable(learning_curves), f, indent=2)

epochs_range = learning_curves['epochs']
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, learning_curves['train_loss'], label='train_loss')
plt.plot(epochs_range, learning_curves['val_loss'], label='val_loss')
plt.legend()
plt.title('Loss')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, learning_curves['train_f1'], label='train_f1')
plt.plot(epochs_range, learning_curves['val_f1'], label='val_f1')
plt.legend()
plt.title('F1')

learning_curves_path = model_dir / 'learning_curves.png'
plt.tight_layout()
plt.savefig(learning_curves_path, dpi=150)
plt.show()

print(f'Checkpoint saved: {checkpoint_path}')
print(f'Model artifacts saved under: {model_dir}')
print(f'Learning curves saved: {learning_curves_path}')

## 9) Grad-CAM de ejemplo (visualización)

In [ ]:
gradcam_dir = model_dir / 'gradcam'
gradcam_dir.mkdir(parents=True, exist_ok=True)
gradcam_path = gradcam_dir / 'gradcam_positive_label1.png'

try:
    result = None
    try:
        result = generate_gradcam_visualization(model, test_dataset, device, target_label=1, save_path=gradcam_path)
    except TypeError:
        result = generate_gradcam_visualization(model, test_dataset, device, target_label=1)

    if isinstance(result, (str, Path)):
        saved_path = Path(result)
        if not saved_path.is_absolute():
            saved_path = gradcam_dir / saved_path
        print(f'Grad-CAM saved: {saved_path}')
    elif hasattr(result, 'save'):
        result.save(gradcam_path)
        print(f'Grad-CAM saved: {gradcam_path}')
    elif hasattr(result, 'savefig'):
        result.savefig(gradcam_path, dpi=150, bbox_inches='tight')
        print(f'Grad-CAM saved: {gradcam_path}')
    elif hasattr(result, 'figure'):
        result.figure.savefig(gradcam_path, dpi=150, bbox_inches='tight')
        print(f'Grad-CAM saved: {gradcam_path}')
    elif isinstance(result, np.ndarray):
        plt.imsave(gradcam_path, result, cmap='viridis')
        print(f'Grad-CAM saved: {gradcam_path}')
    else:
        if result is None:
            print(f'Grad-CAM generated and saved to: {gradcam_path}')
        else:
            print('Grad-CAM generated, pero no se pudo inferir cómo guardarlo automáticamente.')
except Exception as e:
    print('Grad-CAM skipped:', e)

## 10) Notas
- Este notebook es una plantilla base para entrenar otros modelos: reemplaza la llamada a `build_resnet18` por otras arquitecturas (ej. `build_efficientnet`, `build_densenet`) y ajusta `CONFIG`.
- Para entrenamiento a gran escala añade checkpointing por época y logging (TensorBoard, Weights & Biases, etc.).